In [127]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI

# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [128]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [129]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *
from utils.file_ops import *

In [130]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel

,add_id,good_match,app_status,institution,location,add_rank,add_field,add_category,deadline_target,add_posting,add_how_apply,other_link,need_cover_letter,need_RS,need_TS,need_diversity,OMER_letter_status,KLAUS_letter_status,WOO_letter_status
0,simon_frazer_1,NaN,generating docs,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,2025-10-01,EJM,EJM,NaN,1,1,1,1,NaN,NaN,NaN
1,macmaster_1,NaN,generating docs,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,2025-10-01,EJM,EJM,NaN,1,1,1,1,NaN,NaN,NaN
2,usydney_1,1.0,generating docs,University of Sydney,"Sydney, Asutralia",postdoc,culture,research,2025-11-17,EJM,EJM,NaN,1,0,0,0,NaN,NaN,NaN
3,nova_lisboa_1,NaN,getting info,University Nova Lisboa,"Cascais, Portugal",assistant,environmental,research,2025-11-17,EJM,EJM,NaN,1,0,0,0,NaN,NaN,NaN
4,will_college_1,NaN,getting info,Williams College,"Williamstown, United States",assistant,applied micro,research,2025-11-10,EJM,https://apply.interfolio.com/171825,NaN,1,0,1,0,NaN,NaN,NaN
5,aist_1,1.0,getting info,Institute For Advanced Study,"Touluse, France",fellowship,NaN,research,2025-11-10,NaN,https://www.linkedin.com/posts/nicholascmartin...,NaN,1,0,0,0,NaN,NaN,NaN
6,waseda_1,1.0,getting info,Waseda University,"Waseda, Japan",assistant,applied micro,research,2025-10-10,JOE,https://www.wasedapse.jp/en/fpse2/eng_input.php,https://www.waseda.jp/fpse/pse/news-en/2025/08...,1,1,1,0,NaN,NaN,NaN
7,penn_state_1,1.0,getting info,Pennsylvania State University,"University Park, United States",assistant,NaN,research,2025-11-10,EJM,https://psu.wd1.myworkdayjobs.com/en-US/PSU_Ac...,NaN,1,0,0,0,NaN,NaN,NaN


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [131]:
# Research Statement template
with open(path_templates / "base_text/research_statement.txt", "r") as f:
    lines = f.readlines()

RA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/teaching_statement.txt", "r") as f:
    lines = f.readlines()

TA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/cover_letter.txt", "r") as f:
    lines = f.readlines()

CL_TEMPLATE = " ".join(lines)

# Figure out what docs are needed to generate

In [132]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "need_cover_letter",
    "need_RS",
    "need_TS",
    "need_diversity",
]

In [133]:
completed_docs = [str(d).split("/")[-1] for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- macmaster_1
- will_college_1
- waseda_1
- penn_state_1
- nova_lisboa_1
- usydney_1
- aist_1
- simon_frazer_1


In [134]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,1
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,1
2,usydney_1,University of Sydney,"Sydney, Asutralia",postdoc,culture,research,1,0,0,0
3,nova_lisboa_1,University Nova Lisboa,"Cascais, Portugal",assistant,environmental,research,1,0,0,0
4,will_college_1,Williams College,"Williamstown, United States",assistant,applied micro,research,1,0,1,0


# Add the context and additional data for each prompt

For these all the `raw` documents are transcribed in a few bullet points:

- For Job add use 
    ```
    summarize the job add in a few bullet points. Prioritize the attributes, qualitites they look for in a candidate
    ```
- For the isntitutional values use 
    ```
    summarize the institution mission and values in a few bullet points
    ```
- For econ research department use 
    ```
    summarize the economics department values and research topics in a few bullet points. Make emphasis on potential co-authors and researcg/teaching groups described there and write it in the file
    ```

In [135]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START JOB DESCRIPTION",
        end_marker="END JOB DESCRIPTION",
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START ECON DEPT",
        end_marker="END ECON DEPT",
    ),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START INSTITUTIONAL DESCRIPTION",
        end_marker="END INSTITUTIONAL DESCRIPTION",
    ),
    axis=1,
)

# OpenAI Batches

In [136]:
openai_client = OpenAI()
model_name = "gpt-4.1-mini"

In [146]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

In-process batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
7,2025-09-29 14:14:25.881755,Generate Teaching Statements,gpt-4.1-mini,ts,batch_68dada9307e08190a5c9b887fe491a8a,in_progress,False,file-AdDyDjdh2ESX9wEnyE97Fo,NaN,NaN


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


In [98]:
# batch_history

In [99]:
class BodyText(BaseModel):
    body_text: str
    cot: str
    model_config = ConfigDict(extra="forbid")

# Cover Letter

In [119]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()
gen_aux

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity,add_description,econ_context,institution_values
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,1,- Tenure-track Assistant Professor position in...,"- The department fosters a dynamic, creative, ...",- Student participation and involvement are ce...
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,1,- Position: Assistant or junior Associate Prof...,- The Department values rigorous training in e...,"- Commitment to creativity, innovation, and ex..."
2,usydney_1,University of Sydney,"Sydney, Asutralia",postdoc,culture,research,1,0,0,0,- PhD in Economics (awarded prior to start)\n-...,- ARC Discovery Project led by faculty at Univ...,- Commitment to education for all and leadersh...
3,nova_lisboa_1,University Nova Lisboa,"Cascais, Portugal",assistant,environmental,research,1,0,0,0,"- Position: Assistant (tenure track), Associat...",- The Economics Department at Nova SBE values ...,"- Freedom of opinion, expression, and promotio..."
4,will_college_1,Williams College,"Williamstown, United States",assistant,applied micro,research,1,0,1,0,- Seeking two tenure-track faculty in applied ...,- The Williams Economics Department values bot...,- Commitment to a liberal arts education foste...
5,aist_1,Institute For Advanced Study,"Touluse, France",fellowship,NaN,research,1,0,0,0,,,
6,waseda_1,Waseda University,"Waseda, Japan",assistant,applied micro,research,1,1,1,0,,,
7,penn_state_1,Pennsylvania State University,"University Park, United States",assistant,NaN,research,1,0,0,0,,,


In [120]:
def cl_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to add a position fit paragraph to the following COVER LETTER START and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    Keep the length of the paragraph to about 100 words.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full new paragraph as `body_text` and a short chain-of-thought explanation of how you modified the BASE COVER LETTER START to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE COVER LETTER START.

    BASE COVER LETTER START:
    {CL_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [121]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: cl_prompt(row),
    axis=1,
)

In [122]:
name_jsonl = "gen_rs"
description_batch = "Generate Cover Letters"

content_schema = BodyText
cat_type = "cl"

In [123]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and CL
Updating the existing old_docs
No CL to generate using gpt-4.1-mini


# Research Statement

In [105]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()
gen_aux

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity,add_description,econ_context,institution_values
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,1,- Tenure-track Assistant Professor position in...,"- The department fosters a dynamic, creative, ...",- Student participation and involvement are ce...
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,1,- Position: Assistant or junior Associate Prof...,- The Department values rigorous training in e...,"- Commitment to creativity, innovation, and ex..."
6,waseda_1,Waseda University,"Waseda, Japan",assistant,applied micro,research,1,1,1,0,,,


In [ ]:
def rs_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE RESEARCH STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my research fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the research statement to about 800 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full research statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE RESEARCH STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE RESEARCH STATEMENT.

    BASE RESEARCH STATEMENT:
    {RA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [107]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: rs_prompt(row),
    axis=1,
)

In [108]:
name_jsonl = "gen_rs"
description_batch = "Generate Research Statements"

content_schema = BodyText
cat_type = "rs"

In [109]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and RS
Updating the existing old_docs
No RS to generate using gpt-4.1-mini


# Teaching Statement

In [139]:
gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()
gen_aux

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity,add_description,econ_context,institution_values
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,1,- Tenure-track Assistant Professor position in...,"- The department fosters a dynamic, creative, ...",- Student participation and involvement are ce...
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,1,- Position: Assistant or junior Associate Prof...,- The Department values rigorous training in e...,"- Commitment to creativity, innovation, and ex..."
4,will_college_1,Williams College,"Williamstown, United States",assistant,applied micro,research,1,0,1,0,- Seeking two tenure-track faculty in applied ...,- The Williams Economics Department values bot...,- Commitment to a liberal arts education foste...
6,waseda_1,Waseda University,"Waseda, Japan",assistant,applied micro,research,1,1,1,0,,,


In [140]:
def ts_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE TEACHING STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my educational philosophy fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the teaching statement to about 800 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full teaching statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE TEACHING STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE TEACHING STATEMENT.

    BASE TEACHING STATEMENT:
    {TA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [141]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: ts_prompt(row),
    axis=1,
)

In [142]:
name_jsonl = "gen_ts"
description_batch = "Generate Teaching Statements"

content_schema = BodyText
cat_type = "ts"

In [143]:
%run -i batched_chat_gpt.py

Need to generate 4 TS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68dada9307e08190a5c9b887fe491a8a and updated the batch history.


# Generate the Docs

Here we assume that all docs have been generated

In [124]:
def generate_docs(df, generated_df, gen_type="rs"):
    if gen_type == "rs":
        dir_name = "research_statement"
        file_name = "Gonzalez_RS.typ"
    elif gen_type == "ts":
        dir_name = "teaching_statement"
        file_name = "Gonzalez_TS.typ"
    elif gen_type == "cl":
        dir_name = "cover_letter"
        file_name = "Gonzalez_CL.typ"
    else:
        raise ValueError("gen_type not recognized")

    for i, row in df.iterrows():
        print(f"Processing {row.add_id}...")

        new_content = generated_df[
            generated_df.add_id == row.add_id
        ].body_text_1.values[0]

        target_path = path_output / row.add_id
        subdir_path = target_path / dir_name

        if not target_path.exists():
            print(f"Creating directory: {target_path}")
            copy_and_rename_dir(path_templates / f"{dir_name}", target_path, dir_name)
        else:
            print(f"Main Directory already exists: {target_path}")
            if not subdir_path.exists():
                print(f"Creating subdirectory: {subdir_path}")
                copy_and_rename_dir(
                    path_templates / f"{dir_name}", target_path, dir_name
                )

        try:
            add_lines_to_typs_doc(subdir_path / file_name, new_content)
            print(f"{file_name} added for {row.add_id}")
        except Exception as e:
            print(f"Error processing {row.add_id}: {e}")

## Cover Letter

In [125]:
cl_gen_df = pd.read_csv(path_output / "cl_docs_gpt-4.1-mini.csv")
cl_gen_df

,add_id,body_text_1,cot_1
0,aist_1,I am particularly drawn to the Institute for A...,To tailor the cover letter to the Institute fo...
1,macmaster_1,In alignment with McMaster University's dedica...,I crafted a paragraph that connects my researc...
2,nova_lisboa_1,My research program aligns closely with the Ec...,I incorporated the institution's core values s...
3,penn_state_1,"At Pennsylvania State University, I am drawn t...",I crafted a position-fit paragraph that connec...
4,simon_frazer_1,My background and research agenda are well ali...,I crafted a paragraph that connects my researc...
5,usydney_1,My research agenda aligns seamlessly with the ...,I incorporated the institution's values by emp...
6,waseda_1,Waseda University's commitment to fostering in...,I crafted a paragraph that explicitly connects...
7,will_college_1,My research agenda and pedagogical philosophy ...,"To tailor the position fit paragraph, I emphas..."


In [126]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()

generate_docs(gen_aux, cl_gen_df, gen_type="cl")

Processing simon_frazer_1...
Main Directory already exists: ../output_docs/simon_frazer_1
Creating subdirectory: ../output_docs/simon_frazer_1/cover_letter
Gonzalez_CL.typ added for simon_frazer_1
Processing macmaster_1...
Main Directory already exists: ../output_docs/macmaster_1
Creating subdirectory: ../output_docs/macmaster_1/cover_letter
Gonzalez_CL.typ added for macmaster_1
Processing usydney_1...
Creating directory: ../output_docs/usydney_1
Gonzalez_CL.typ added for usydney_1
Processing nova_lisboa_1...
Creating directory: ../output_docs/nova_lisboa_1
Gonzalez_CL.typ added for nova_lisboa_1
Processing will_college_1...
Creating directory: ../output_docs/will_college_1
Gonzalez_CL.typ added for will_college_1
Processing aist_1...
Creating directory: ../output_docs/aist_1
Gonzalez_CL.typ added for aist_1
Processing waseda_1...
Main Directory already exists: ../output_docs/waseda_1
Creating subdirectory: ../output_docs/waseda_1/cover_letter
Gonzalez_CL.typ added for waseda_1
Process

## Research Statement

In [116]:
rs_gen_df = pd.read_csv(path_output / "rs_docs_gpt-4.1-mini.csv")
rs_gen_df

,add_id,body_text_1,cot_1
0,macmaster_1,"Entertainment media, encompassing television s...",I integrated the institution's values and depa...
1,simon_frazer_1,# Research Statement\n\nEntertainment media su...,I refined the base research statement to align...
2,waseda_1,# Research Statement\n\nEntertainment media—in...,I integrated the core content of the base rese...


In [117]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()

generate_docs(gen_aux, rs_gen_df, gen_type="rs")

Processing simon_frazer_1...
Creating directory: ../output_docs/simon_frazer_1
Gonzalez_RS.typ added for simon_frazer_1
Processing macmaster_1...
Creating directory: ../output_docs/macmaster_1
Gonzalez_RS.typ added for macmaster_1
Processing waseda_1...
Creating directory: ../output_docs/waseda_1
Gonzalez_RS.typ added for waseda_1
